In [1]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
from datetime import date
import pandas as pd
import os

today = datetime.today()
current_year = today.year
print(current_year)
last_month = today - relativedelta(months=1)
last_month_year = last_month.year
last_month = last_month.strftime('%B')

2026


##### Combine Hours Data

In [2]:
# Set the folder path containing CSV files
folder_path = r'C:\Users\IlaBarshilia\ACME Barricades\Acme Safety Team - Safety Dashboard Inputs\Hours Worked Reports\\' + str(last_month_year)

# List to hold individual DataFrames
df_list = []

# Loop through all excel files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.xlsx') or filename.endswith('.XLSX'):
        file_path = os.path.join(folder_path, filename)
        
        # Read the CSV file
        df = pd.read_excel(file_path)
        
        # Find the row where 'Employee' occurs in the first column
        header_row_idx = df.index[df.iloc[:, 0].astype(str).str.contains("Employee", case=False, na=False)].tolist()
        
        if header_row_idx:
            header_row = header_row_idx[0]
        
            # Promote this row as header
            df.columns = df.iloc[header_row]
            
            # Remove rows above header
            df = df.iloc[header_row + 1:].reset_index(drop=True)

        # Extract text after the last '-' in the filename (excluding extension)
        label = filename.lower().rsplit(' ', 1)[-1].replace('.xlsx', '')
        df['Date'] = label
        # Append to the list
        df_list.append(df)

# Combine all DataFrames into one
final_df = pd.concat(df_list, ignore_index=True)

search_text = "Totals for Location:"
# Find all matching indices
match_idx = final_df.index[final_df['Employee'].str.contains(search_text, case=False, na=False)].tolist()

# Copy text to next row and remove original row
for i in reversed(match_idx):  # reverse to avoid index shift issues
    if i + 1 < len(final_df):
        final_df.at[i + 1, 'Employee'] = final_df.at[i, 'Employee']
    final_df = final_df.drop(i)

# Reset index after removal
final_df = final_df.reset_index(drop=True)

# Keep only rows where column contains the value
filtered_df = final_df[final_df['Employee'].str.contains(search_text, case=False, na=False)]
filtered_df = filtered_df[['Employee', 'Total Hrs', 'Date']]
filtered_df.rename(columns = {'Employee':'Location', 'Total Hrs':'Hours'}, inplace=True)
filtered_df['Location'] = filtered_df['Location'].str.replace('Totals for Location:', '',regex=False)
filtered_df['Sort Order'] = filtered_df['Date'].str.split('-', n=1).str[0]
filtered_df = filtered_df.drop_duplicates()
filtered_df.to_excel(r'C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\Combined_Hours_Data.xlsx', index=False)

##### Combine Geotab Reports

In [3]:
geotab_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\Advanced Trips Summary Report GEOTAB.xlsx", sheet_name="Report")

### Use below module if geotab data doesn't strat from first row
# expected_headers = ["Name", "Group", "Period Start Date", "Driving Duration"]
# header_row_idx = None

# # --- Find the row where all expected headers appear somewhere in the row ---
# for i in range(len(geotab_df)):
#     row_values = [x for x in geotab_df.iloc[i].tolist()]
#     # Check if every expected header is present in this row (order not required)
#     if all(h in row_values for h in expected_headers):
#         header_row_idx = i
#         break

# if header_row_idx is None:
#     raise ValueError("Header row not found. Check expected_headers or file content.")

# # --- Promote that row to columns ---
# new_columns = [str(x).strip() for x in geotab_df.iloc[header_row_idx].tolist()]
# geotab_df = geotab_df.iloc[header_row_idx + 1:].reset_index(drop=True)
# geotab_df.columns = new_columns

# # Optional cleanup: drop columns that are entirely NaN after promotion
# geotab_df = geotab_df.dropna(axis=1, how="all")

folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\\" + str(last_month_year) + "\\" + last_month
geotab_search_text = "Advanced Trips Summary Report"
expected_headers = ["Name", "Group", "Period Start Date", "Driving Duration"]
for filename in os.listdir(folder_path):
    if geotab_search_text in filename:
        file_path = os.path.join(folder_path, filename)
        latest_geotab_df = pd.read_excel(file_path, sheet_name = "Report")
        header_row_idx = None
        for i in range(len(latest_geotab_df)):
            row_values = [x for x in latest_geotab_df.iloc[i].tolist()]
            # Check if every expected header is present in this row (order not required)
            if all(h in row_values for h in expected_headers):
                header_row_idx = i
                break

        if header_row_idx is None:
            raise ValueError("Header row not found. Check expected_headers or file content.")

        # --- Promote that row to columns ---
        new_columns = [str(x).strip() for x in latest_geotab_df.iloc[header_row_idx].tolist()]
        latest_geotab_df = latest_geotab_df.iloc[header_row_idx + 1:].reset_index(drop=True)
        latest_geotab_df.columns = new_columns

        # Optional cleanup: drop columns that are entirely NaN after promotion
        latest_geotab_df = latest_geotab_df.dropna(axis=1, how="all")

geotab_df = pd.concat([geotab_df, latest_geotab_df])
geotab_df = geotab_df.drop_duplicates()

geotab_df.to_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\Advanced Trips Summary Report GEOTAB.xlsx", sheet_name="Report", index=False)

##### Combine Advanced Inspection Report from Geotab

In [10]:
asset_insp_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\Advanced Asset Inspection Report.xlsx", sheet_name="Report")
# asset_insp_df.head(15)
# ### Use below module if asset_inspection data doesn't start from first row
# expected_headers = ["Driver", "First Name", "Last Name", "Device", 'Log type']
# header_row_idx = None

# # --- Find the row where all expected headers appear somewhere in the row ---
# for i in range(len(asset_insp_df)):
#     row_values = [x for x in asset_insp_df.iloc[i].tolist()]
#     # Check if every expected header is present in this row (order not required)
#     if all(h in row_values for h in expected_headers):
#         header_row_idx = i
#         break

# if header_row_idx is None:
#     raise ValueError("Header row not found. Check expected_headers or file content.")

# # --- Promote that row to columns ---
# new_columns = [str(x).strip() for x in asset_insp_df.iloc[header_row_idx].tolist()]
# asset_insp_df = asset_insp_df.iloc[header_row_idx + 1:].reset_index(drop=True)
# asset_insp_df.columns = new_columns
# print('Shape of entire asset inspection df', asset_insp_df.shape)
# asset_insp_df = asset_insp_df[['Reporting Date','Device', 'Duration']].drop_duplicates()
# print('Shape after keeping only relevant columns', asset_insp_df.shape)
# asset_insp_df

folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\\" + str(last_month_year) + "\\" + last_month
asset_insp_search_text = "Advanced Asset Inspection Report_20260415_100008"
expected_headers = ["Driver", "First Name", "Last Name", "Device"]
for filename in os.listdir(folder_path):
    if asset_insp_search_text in filename:
        file_path = os.path.join(folder_path, filename)
        latest_asset_insp_df = pd.read_excel(file_path, sheet_name = "Report")
        header_row_idx = None
        for i in range(len(latest_asset_insp_df)):
            row_values = [x for x in latest_asset_insp_df.iloc[i].tolist()]
            # Check if every expected header is present in this row (order not required)
            if all(h in row_values for h in expected_headers):
                header_row_idx = i
                break

        if header_row_idx is None:
            raise ValueError("Header row not found. Check expected_headers or file content.")

        # --- Promote that row to columns ---
        new_columns = [str(x).strip() for x in latest_asset_insp_df.iloc[header_row_idx].tolist()]
        latest_asset_insp_df = latest_asset_insp_df.iloc[header_row_idx + 1:].reset_index(drop=True)
        latest_asset_insp_df.columns = new_columns

        # Optional cleanup: drop columns that are entirely NaN after promotion
        latest_asset_insp_df = latest_asset_insp_df.dropna(axis=1, how="all")
        latest_asset_insp_df = latest_asset_insp_df[['Reporting Date','Device', 'Duration', 'Log type']].drop_duplicates()

asset_insp_df_df = pd.concat([asset_insp_df, latest_asset_insp_df])
asset_insp_df = asset_insp_df.drop_duplicates()
print('Shape after merging with last months data', asset_insp_df.shape)
asset_insp_df.to_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\Advanced Asset Inspection Report.xlsx", sheet_name="Report", index=False)

Shape after merging with last months data (917, 3)


##### Combine Fleetio Inspection History Reports

In [4]:
fleetio_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\fleetio-inspection-history-export.xlsx")
print("Shape before merging this month's data:", fleetio_df.shape)
fleetio_df['Submitted On (Eastern Time (US & Canada))'] = pd.to_datetime(fleetio_df['Submitted On (Eastern Time (US & Canada))'])
folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\\" + str(last_month_year) + "\\" + str(last_month)
fleetio_search_text = "fleetio-inspection-history"

for filename in os.listdir(folder_path):
    if fleetio_search_text in filename:
        file_path = os.path.join(folder_path, filename)
        latest_fleetio_df = pd.read_excel(file_path)

fleetio_df = pd.concat([fleetio_df, latest_fleetio_df])
fleetio_df = fleetio_df.drop_duplicates()
fleetio_df['Submitted On (Eastern Time (US & Canada))'] = pd.to_datetime(fleetio_df['Submitted On (Eastern Time (US & Canada))'])
print("Shape after merging this month's data:", fleetio_df.shape)

fleetio_df.to_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\fleetio-inspection-history-export.xlsx", index=False)

Shape before merging this month's data: (12460, 9)
Shape after merging this month's data: (19130, 9)


##### Combine Driver's Alert Data

In [5]:
driveralert_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\DriversAlert.xlsx")
print(driveralert_df.shape)

today = pd.Timestamp.today().normalize()  # current local date
first_of_this_month = today.replace(day=1)
last_month_end = first_of_this_month - pd.Timedelta(days=1)
last_month_start = last_month_end.replace(day=1)

# driveralert_df['dateassigned'] = pd.to_datetime(driveralert_df['dateassigned'], format="%m/%d/%Y %I:%M:%S %p" , errors='coerce')
# driveralert_df = driveralert_df[(driveralert_df['dateassigned'] >= pd.Timestamp("2025-01-01")) & (driveralert_df['dateassigned'] <= pd.Timestamp("2025-10-31"))]

current_month = (datetime.now().replace(day=1)).strftime('%B')
folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\\" + str(last_month_year) + "\\" + str(last_month)
da_search_text = "driversalert"

for filename in os.listdir(folder_path):
    if da_search_text.lower() in filename.lower():
        file_path = os.path.join(folder_path, filename)
        latest_driveralert_df = pd.read_excel(file_path)

latest_driveralert_df['dateassigned'] = pd.to_datetime(latest_driveralert_df['dateassigned'], format="%m/%d/%Y %I:%M:%S %p" , errors='coerce')
latest_driveralert_df = latest_driveralert_df[(latest_driveralert_df['dateassigned'] >= last_month_start) & (latest_driveralert_df['dateassigned'] < first_of_this_month)]

driveralert_df = pd.concat([driveralert_df, latest_driveralert_df])
driveralert_df = driveralert_df.drop_duplicates()
print(driveralert_df.shape)
driveralert_df.to_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\DriversAlert.xlsx", index=False)

(911, 21)


c:\Users\IlaBarshilia\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


(1388, 21)


##### Combine SBN Data

In [6]:
SBN_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\Output_SBN_YTD.xlsx")
print(SBN_df.shape)

today = pd.Timestamp.today().normalize()  # current local date
first_of_this_month = today.replace(day=1)
last_month_end = first_of_this_month - pd.Timedelta(days=1)
last_month_start = last_month_end.replace(day=1)

current_month = (datetime.now().replace(day=1)).strftime('%B')
folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\\" + str(last_month_year) + "\\" + str(last_month)
sbn_search_text = "Inspection-CSV-Report"

for filename in os.listdir(folder_path):
    if sbn_search_text.lower() in filename.lower():
        file_path = os.path.join(folder_path, filename)
        latest_SBN_df = pd.read_csv(file_path)
        latest_SBN_df['Month - Year'] = last_month_start

SBN_df = pd.concat([SBN_df, latest_SBN_df])
SBN_df = SBN_df.drop_duplicates()
print(SBN_df.shape)
SBN_df.to_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\Output_SBN_YTD.xlsx", index=False)

(42, 3)
(67, 3)


#### Combine Fleetio Issue Export Data

In [7]:
fleetio_issue_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\fleetio-issue-export.xlsx")
print("Shape of fleetio issue df before merging this month's data:", fleetio_issue_df.shape)

today = pd.Timestamp.today().normalize()  # current local date
first_of_this_month = today.replace(day=1)
last_month_end = first_of_this_month - pd.Timedelta(days=1)
last_month_start = last_month_end.replace(day=1)

current_month = (datetime.now().replace(day=1)).strftime('%B')
folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\\" + str(last_month_year) + "\\" + str(last_month)
fl_issue_search_text = "fleetio-issue-export"

for filename in os.listdir(folder_path):
    if fl_issue_search_text.lower() in filename.lower():
        file_path = os.path.join(folder_path, filename)
        latest_fleetio_issue_df = pd.read_excel(file_path)
        latest_fleetio_issue_df['Month - Year'] = last_month_start
        print("Shape before Active Asset Status and open Issue Status:", latest_fleetio_issue_df.shape)
        latest_fleetio_issue_df = latest_fleetio_issue_df[(latest_fleetio_issue_df['Asset Status'] == "Active") & (latest_fleetio_issue_df['Issue Status'] == "Open")]
        print("Shape after Active Asset Status and open Issue Status:", latest_fleetio_issue_df.shape)

fleetio_issue_df = pd.concat([fleetio_issue_df, latest_fleetio_issue_df])
fleetio_issue_df = fleetio_issue_df.drop_duplicates()
print("Shape of fleetio issue df after merging this month's data:", fleetio_issue_df.shape)
fleetio_issue_df.to_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\fleetio-issue-export.xlsx", index=False)

Shape of fleetio issue df before merging this month's data: (193, 15)
Shape before Active Asset Status and open Issue Status: (85, 15)
Shape after Active Asset Status and open Issue Status: (85, 15)
Shape of fleetio issue df after merging this month's data: (278, 15)


#### Combine Service Remindear Data


In [8]:
fleetio_service_reminder_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\fleetio-service-reminder-export.xlsx")
print("Shape of fleetio service reminder df before merging this month's data:", fleetio_service_reminder_df.shape)

today = pd.Timestamp.today().normalize()  # current local date
first_of_this_month = today.replace(day=1)
last_month_end = first_of_this_month - pd.Timedelta(days=1)
last_month_start = last_month_end.replace(day=1)

current_month = (datetime.now().replace(day=1)).strftime('%B')
folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\\" + str(last_month_year) + "\\" + str(last_month)
fl_service_search_text = "fleetio-service-reminder"

for filename in os.listdir(folder_path):
    if fl_service_search_text.lower() in filename.lower():
        file_path = os.path.join(folder_path, filename)
        latest_fleetio_service_reminder_df = pd.read_excel(file_path)
        latest_fleetio_service_reminder_df['Month - Year'] = last_month_start
        print("\nShape before Active Overdue service Status and active vehicle Status:", latest_fleetio_service_reminder_df.shape)
        latest_fleetio_service_reminder_df = latest_fleetio_service_reminder_df[(latest_fleetio_service_reminder_df['Status'] == "Overdue") & (latest_fleetio_service_reminder_df['Vehicle Status'] == "Active")]
        print("Shape after Active Overdue service Status and active vehicle Status:", latest_fleetio_service_reminder_df.shape)


fleetio_service_reminder_df = pd.concat([fleetio_service_reminder_df, latest_fleetio_service_reminder_df])
fleetio_service_reminder_df = fleetio_service_reminder_df.drop_duplicates()
print("\nShape of fleetio service reminder df after merging this month's data:", fleetio_service_reminder_df.shape)
fleetio_service_reminder_df.to_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\fleetio-service-reminder-export.xlsx", index=False)

Shape of fleetio service reminder df before merging this month's data: (122, 19)

Shape before Active Overdue service Status and active vehicle Status: (1505, 19)
Shape after Active Overdue service Status and active vehicle Status: (51, 19)

Shape of fleetio service reminder df after merging this month's data: (173, 19)


#### Combine Vehicle Renewal Reminder's Data

In [9]:
fleetio_veh_renewal_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\fleetio-veh-renewal_reminder.xlsx")
print("Shape of fleetio vehicle renewal df before merging this month's data:", fleetio_veh_renewal_df.shape)

today = pd.Timestamp.today().normalize()  # current local date
first_of_this_month = today.replace(day=1)
last_month_end = first_of_this_month - pd.Timedelta(days=1)
last_month_start = last_month_end.replace(day=1)

current_month = (datetime.now().replace(day=1)).strftime('%B')
folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\\" + str(last_month_year) + "\\" + str(last_month)
fl_veh_renewal_search_text = "vehicle-renewal-reminder"

for filename in os.listdir(folder_path):
    if fl_veh_renewal_search_text.lower() in filename.lower():
        file_path = os.path.join(folder_path, filename)
        latest_fleetio_veh_renewal_df = pd.read_excel(file_path)
        latest_fleetio_veh_renewal_df['Month - Year'] = last_month_start
        print("\nShape before Overdue renewal Status and active vehicle Status:", latest_fleetio_veh_renewal_df.shape)
        latest_fleetio_veh_renewal_df = latest_fleetio_veh_renewal_df[(latest_fleetio_veh_renewal_df['Status'] == "Overdue") & (latest_fleetio_veh_renewal_df['Vehicle Status'] == "Active")]
        print("\nShape after Overdue renewal Status and active vehicle Status:", latest_fleetio_veh_renewal_df.shape)

fleetio_veh_renewal_df = pd.concat([fleetio_veh_renewal_df, latest_fleetio_veh_renewal_df])
fleetio_veh_renewal_df = fleetio_veh_renewal_df.drop_duplicates()
print("\nShape of fleetio vehicle renewal df after merging this month's data:", fleetio_veh_renewal_df.shape)
fleetio_veh_renewal_df.to_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\fleetio-veh-renewal_reminder.xlsx", index=False)

Shape of fleetio vehicle renewal df before merging this month's data: (1, 9)

Shape before Overdue renewal Status and active vehicle Status: (172, 9)

Shape after Overdue renewal Status and active vehicle Status: (4, 9)

Shape of fleetio vehicle renewal df after merging this month's data: (5, 9)


#### Combine Service Compliance Data

In [10]:
fleetio_service_comp_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\fleetio-service-compliance.xlsx")
print("Shape of fleetio service compliance df before merging this month's data:", fleetio_service_comp_df.shape)

today = pd.Timestamp.today().normalize()  # current local date
first_of_this_month = today.replace(day=1)
last_month_end = first_of_this_month - pd.Timedelta(days=1)
last_month_start = last_month_end.replace(day=1)

current_month = (datetime.now().replace(day=1)).strftime('%B')
folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\\" + str(last_month_year) + "\\" + str(last_month)
fl_service_comp_search_text = "service reminder compliance"

for filename in os.listdir(folder_path):
    if fl_service_comp_search_text.lower() in filename.lower():
        file_path = os.path.join(folder_path, filename)
        latest_fleetio_service_comp_df = pd.read_csv(file_path)
        latest_fleetio_service_comp_df['Month - Year'] = last_month_start

fleetio_service_comp_df = pd.concat([fleetio_service_comp_df, latest_fleetio_service_comp_df])
fleetio_service_comp_df = fleetio_service_comp_df.drop_duplicates()
print("\nShape of fleetio service compliance df after merging this month's data:", fleetio_service_comp_df.shape)
fleetio_service_comp_df.to_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\fleetio-service-compliance.xlsx", index=False)

Shape of fleetio service compliance df before merging this month's data: (73, 4)

Shape of fleetio service compliance df after merging this month's data: (110, 4)


#### Combine Watchdog reports from Geotab

In [11]:
watchdog_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\Watchdog Report.xlsx")
print("Shape of watchdog df before merging this month's data:", watchdog_df.shape)

today = pd.Timestamp.today().normalize()  # current local date
first_of_this_month = today.replace(day=1)
last_month_end = first_of_this_month - pd.Timedelta(days=1)
last_month_start = last_month_end.replace(day=1)

current_month = (datetime.now().replace(day=1)).strftime('%B')
folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\\" + str(last_month_year) + "\\" + str(last_month)
watchdog_search_text = "Watchdog Report"

for filename in os.listdir(folder_path):
    if watchdog_search_text.lower() in filename.lower():
        file_path = os.path.join(folder_path, filename)
        latest_watchdog_df = pd.read_excel(file_path)

## Use below module if watchdog data doesn't strat from first row
expected_headers = ["Vehicle", "Group", "Device ID", "Status"]
header_row_idx = None

# --- Find the row where all expected headers appear somewhere in the row ---
for i in range(len(latest_watchdog_df)):
    row_values = [x for x in latest_watchdog_df.iloc[i].tolist()]
    # Check if every expected header is present in this row (order not required)
    if all(h in row_values for h in expected_headers):
        header_row_idx = i
        break

if header_row_idx is None:
    raise ValueError("Header row not found. Check expected_headers or file content.")

# --- Promote that row to columns ---
new_columns = [str(x).strip() for x in latest_watchdog_df.iloc[header_row_idx].tolist()]
latest_watchdog_df = latest_watchdog_df.iloc[header_row_idx + 1:].reset_index(drop=True)
latest_watchdog_df.columns = new_columns
latest_watchdog_df['Last Communication Date'] = pd.to_datetime(latest_watchdog_df['Last Communication Date'], errors='coerce')
latest_watchdog_df = latest_watchdog_df[(latest_watchdog_df['Last Communication Date'] >= pd.Timestamp.today().replace(day=1) - pd.DateOffset(months=1)) &
    (latest_watchdog_df['Last Communication Date'] < pd.Timestamp.today().replace(day=1))]

watchdog_df = pd.concat([watchdog_df, latest_watchdog_df])
watchdog_df = watchdog_df.drop_duplicates()
print("\nShape of watchdog report df after merging this month's data:", watchdog_df.shape)
watchdog_df.to_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\Watchdog Report.xlsx", index=False)

Shape of watchdog df before merging this month's data: (20, 18)

Shape of watchdog report df after merging this month's data: (55, 18)


c:\Users\IlaBarshilia\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


#### Combine Vehicle Status Summary from Fleetio

In [12]:
veh_status_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\vehicle status summary.xlsx")
print("Shape of vehicle status summary df before merging this month's data:", veh_status_df.shape)

today = pd.Timestamp.today().normalize()  # current local date
first_of_this_month = today.replace(day=1)
last_month_end = first_of_this_month - pd.Timedelta(days=1)
last_month_start = last_month_end.replace(day=1)

current_month = (datetime.now().replace(day=1)).strftime('%B')
folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\\" + str(last_month_year) + "\\" + str(last_month)
veh_status_search_text = "vehicle status summary"

for filename in os.listdir(folder_path):
    if veh_status_search_text.lower() in filename.lower():
        file_path = os.path.join(folder_path, filename)
        latest_veh_status_df = pd.read_csv(file_path)
        latest_veh_status_df['Month - Year'] = last_month_start

veh_status_df = pd.concat([veh_status_df, latest_veh_status_df])
veh_status_df = veh_status_df.drop_duplicates()
print("\nShape of vehicle status summary df after merging this month's data:", veh_status_df.shape)
veh_status_df.to_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Safety Data Dashboard\vehicle status summary.xlsx", index=False)

Shape of vehicle status summary df before merging this month's data: (1297, 12)

Shape of vehicle status summary df after merging this month's data: (1960, 12)
